# Stage 1: Representation Learning (SimCLR)

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [4]:
# Check if GPU is enabled
print(torch.cuda.is_available())  # should print True
print(torch.cuda.get_device_name(0))  # should print the GPU name

True
Tesla T4


In [5]:
# Check GPU memory (to check if 512 batch size is viable)
print(torch.cuda.get_device_name(0))
print(f"Memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4
Memory available: 15.6 GB


In [6]:
# Mount Google Drive so checkpoints can be saved
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# Define augmentation pipeline according to SimCLR paper

transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [8]:
# Define a wrapper class essentially so augmentation is applied twice -> results in two independent views per image.

class TwoViewDataset:
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        img, label = self.base_dataset[index]
        view1 = self.transform(img)
        view2 = self.transform(img)
        return (view1, view2), label

In [19]:
# Load base CIFAR10
base_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)

# Wrap it -> create trainset and its DataLoader!
trainset = TwoViewDataset(base_dataset=base_cifar10, transform=transform)
trainloader = DataLoader(trainset, batch_size=512, shuffle=True, num_workers=2)

 12%|█▏        | 20.3M/170M [00:10<01:15, 1.98MB/s]


KeyboardInterrupt: 

In [9]:
import torch.nn as nn

In [10]:
# First load the ResNet-18 classifier
resnet18 = torchvision.models.resnet18(weights=None, progress=True)

# Reduce kernel size and stride as CIFAR10 images are very small
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

# Remove max pooling layer for same reason
resnet18.maxpool = nn.Identity()

# Remove final classification layer (as embeddings are in penultimate layer)
resnet18.fc = nn.Identity()

In [11]:
# Define projection head (only used during training)
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512, 512)
        self.bn = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(512, 128)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.fc2(x)
        return F.normalize(x, dim=1)

In [12]:
# Define SimCLR model (a wrapper of the ResNet-18 and projection head)

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18
        self.projection = ProjectionHead()

    def forward(self, x1, x2):
        h1 = self.encoder(x1)  # (batch_size, 512)
        h2 = self.encoder(x2)  # (batch_size, 512)

        z1 = self.projection(h1)  # (batch_size, 128)
        z2 = self.projection(h2)  # (batch_size, 128)

        return z1, z2

    # Use after training for embedding extraction
    def get_embedding(self, x):
        return self.encoder(x)


In [16]:
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
      out = torch.cat([z1, z2], dim=0)
      n_samples = len(out)

      # Full similarity matrix
      cov = torch.mm(out, out.t().contiguous())
      sim = torch.exp(cov / self.temperature)

      mask = ~torch.eye(n_samples, device=sim.device).bool()
      neg = sim.masked_select(mask).view(n_samples, -1).sum(dim=-1)

      # Positive similarity
      pos = torch.exp(torch.sum(z1 * z2, dim=-1) / self.temperature)
      pos = torch.cat([pos, pos], dim=0)

      loss = -torch.log(pos / neg).mean()
      return loss

In [17]:
# Instantiate requirements for training loop

model = SimCLR().to(device)
num_epochs = 500
optimiser = torch.optim.SGD(model.parameters(), lr=0.4, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=num_epochs)
criterion = NTXentLoss(temperature=0.5).to(device)

In [20]:
# Continue model training

checkpoint_path = '/content/drive/MyDrive/5CCSAMLF_CW2/models/simclr_latest.pth'

checkpoint = torch.load(checkpoint_path)

model.load_state_dict(checkpoint['model_state_dict'])
optimiser.load_state_dict(checkpoint['optimiser_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1  # resume from next epoch

print(f"Resuming from epoch {start_epoch}, last loss: {checkpoint['loss']:.4f}")


Resuming from epoch 500, last loss: 5.0962


In [ ]:
# Train the model

model.train()
for epoch in range(start_epoch, num_epochs):
    total_loss = 0

    for (x1, x2), labels in trainloader:
        x1, x2 = x1.to(device), x2.to(device)

        optimiser.zero_grad()

        z1, z2 = model(x1, x2)
        loss = criterion(z1, z2)

        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(trainloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

    # Save checkpoint periodically
    if (epoch + 1) % 50 == 0:
      torch.save({'epoch': epoch,
                  'model_state_dict': model.state_dict(),
                  'optimiser_state_dict': optimiser.state_dict(),
                  'scheduler_state_dict': scheduler.state_dict(),
                  'loss': avg_loss},
                  checkpoint_path)
      print(f"Model checkpoint at epoch {epoch+1}")

Step 4: extract embeddings of all 50k CIFAR10 images

In [21]:
# Load trained model
checkpoint = torch.load(checkpoint_path)
model = SimCLR().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval() # not training anymore

# Use plain CIFAR-10 with NO augmentation for extraction
plain_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
plain_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True,
                                              download=True, transform=plain_transform)
plain_loader = DataLoader(plain_cifar10, batch_size=512, shuffle=False)

100%|██████████| 170M/170M [01:00<00:00, 2.81MB/s]


In [ ]:
# Extract all embeddings
all_embeddings = []
all_labels = []

with torch.no_grad():
    for imgs, labels in plain_loader:
        imgs = imgs.to(device)
        emb = model.get_embedding(imgs)         # (batch, 512)
        emb = F.normalize(emb, dim=1)           # L2 normalise
        all_embeddings.append(emb.cpu())
        all_labels.append(labels)
        print(f"Processed {len(all_embeddings) * 512} images")

embeddings = torch.cat(all_embeddings, dim=0)   # (50000, 512)
labels = torch.cat(all_labels, dim=0)           # (50000,) - ground truth, only used for evaluation

In [23]:
# Save embeddings
torch.save({
    'embeddings': embeddings,  # (50000, 512) tensor
    'labels': labels           # (50000,) tensor
}, '/content/drive/MyDrive/5CCSAMLF_CW2/models/embeddings.pth')

# Stage 2: Clustering for diversity

K = min(|Li−1|+ B,max_clusters)

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans
import numpy as np

def cluster_embeddings(embeddings, K):
    """
    embeddings: numpy array of shape (N, 512)
    K: number of clusters = |L_prev| + B
    returns: cluster assignment for each of the N points
    """
    # Paper uses KMeans for K<=50, MiniBatchKMeans otherwise (Appendix F.1)
    if K <= 50:
        kmeans = KMeans(n_clusters=K, random_state=42)
    else:
        kmeans = MiniBatchKMeans(n_clusters=K, random_state=42)

    cluster_assignments = kmeans.fit_predict(embeddings)  # (N,) array
    return cluster_assignments

# Stage 3 - Querying for typicality + diversity

Step 1: define typicality function

> To capture the principle of max density, we define the Typicality of an example by its density in some semantically meaningful feature space. Formally, we measure an example’s Typicality by the inverse of the average Euclidean distance to its K* nearest neighbours, namely:

\text{Typicality}(x) = \left(\frac{1}{K}\sum_{x_i \in \text{K-NN}(x)} \|x - x_i\|_2\right)^{-1}

*K = 20 was used in the paper




In [ ]:
from sklearn.neighbors import NearestNeighbors

def Typicality(cluster_embeddings, K=20):
    """
    cluster_embeddings: (M, 512) array of embeddings for points in ONE cluster
    K: number of nearest neighbours, paper uses K=20
    returns: (M,) array of typicality scores, one per point
    """

    # Fit KNN on the cluster embeddings
    # K+1 because each point is its own nearest neighbour
    k = min(K + 1, len(cluster_embeddings))
    neighbours = NearestNeighbors(n_neighbors=k).fit(cluster_embeddings)
    distances, _ = neighbours.kneighbors(cluster_embeddings)

    # distances[:, 0] is always 0 (self), so skip it
    avg_distances = distances[:, 1:].mean(axis=1)  # (M,)

    typicality = 1.0 / avg_distances  # inverse of average distance
    return typicality